[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vkoul/advanced-ab-testing/blob/main/study_notes/week07_multiple_testing.ipynb)

# Week 7: Multiple Testing & Experiment Infrastructure

## Overview

At QuickCart, our experimentation platform runs dozens of tests simultaneously, each measuring 15-20 metrics. This week we tackle two problems that arise at scale:

1. **Multiple Testing Corrections** -- When you test many metrics, false positives pile up. We study three correction methods (Bonferroni, Holm, Benjamini-Hochberg) and learn when to use each.

2. **Experiment Infrastructure -- User Splitting** -- How do we assign millions of users to dozens of concurrent experiments without cross-contamination? We build a hash-based splitting system with conflict-aware slot allocation.

### Learning Objectives

- Understand why running multiple tests inflates your false positive rate
- Distinguish FWER control from FDR control and choose appropriately
- Implement Bonferroni, Holm, and Benjamini-Hochberg corrections from scratch
- Design a hash-based user splitting system for concurrent experiments
- Handle experiment conflicts via slot allocation

In [ ]:
import numpy as np
import pandas as pd
import hashlib
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations

# Reproducibility
np.random.seed(42)

# Plot style
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

---

# Part 1: Multiple Testing Corrections

## The Problem: Why Testing Many Metrics Is Dangerous

Imagine QuickCart runs an experiment on a new checkout layout. We measure 20 metrics: conversion rate, revenue per user, cart abandonment, page load time, clicks on upsells, and so on. Suppose the new layout has **no real effect** on any metric.

Each individual test uses $\alpha = 0.05$, meaning a 5% chance of a false positive per metric. But across 20 metrics:

$$P(\text{at least one false positive}) = 1 - (1 - 0.05)^{20} \approx 0.64$$

We have a **64% chance** of declaring something significant even when nothing changed. This is the multiple testing problem.

:::{admonition} Full derivation (click to expand)
:class: dropdown

If we run $m$ independent tests, each at significance level $\alpha$, the probability that **none** produce a false positive is:

$$P(\text{no false positives}) = (1 - \alpha)^m$$

Therefore the probability of at least one false positive (the Family-Wise Error Rate under independence) is:

$$\text{FWER} = 1 - (1 - \alpha)^m$$

For $m = 20$ and $\alpha = 0.05$:
$$\text{FWER} = 1 - 0.95^{20} = 1 - 0.3585 \approx 0.64$$

Even with $m = 5$ metrics: $\text{FWER} \approx 0.23$.
:::

In [ ]:
# Demonstrate the multiple testing problem
# Simulate testing 20 null metrics (no real effect) many times

n_simulations = 10000
n_metrics = 20
alpha = 0.05
sample_size = 1000

false_positive_counts = []

for _ in range(n_simulations):
    p_values = []
    for _ in range(n_metrics):
        # Both groups drawn from same distribution (null is true)
        control = np.random.normal(0, 1, sample_size)
        treatment = np.random.normal(0, 1, sample_size)
        _, p = stats.ttest_ind(control, treatment)
        p_values.append(p)
    # Count how many metrics are "significant"
    false_positive_counts.append(sum(p < alpha for p in p_values))

any_fp_rate = np.mean([c > 0 for c in false_positive_counts])
avg_fp = np.mean(false_positive_counts)

print(f"Testing {n_metrics} null metrics at alpha={alpha}:")
print(f"  P(at least one false positive) = {any_fp_rate:.3f}  (theoretical: {1 - (1-alpha)**n_metrics:.3f})")
print(f"  Average false positives per experiment = {avg_fp:.2f}  (expected: {n_metrics * alpha:.1f})")

## Family-Wise Error Rate (FWER)

The **FWER** is the probability of making at least one Type I error (false positive) across all tests:

$$\text{FWER} = P(\text{at least one false rejection of a true null})$$

FWER control is **strict**: it ensures that the probability of even a single false positive stays below $\alpha$. This is appropriate for **guardrail metrics** -- things like crash rate, latency, or revenue where even one false conclusion could be costly.

### Bonferroni Correction

The simplest approach: divide the significance level by the number of tests.

$$\alpha_{\text{adj}} = \frac{\alpha}{m}$$

Reject $H_0$ for test $i$ only if $p_i < \alpha / m$.

**Intuition**: If each test has only an $\alpha/m$ chance of a false positive, then by the union bound, the chance of *any* false positive is at most $m \cdot (\alpha/m) = \alpha$.

**Pros**: Simple, valid regardless of dependence between tests.  
**Cons**: Very conservative -- with many tests, you need extremely small p-values to reject.

:::{admonition} Full derivation (click to expand)
:class: dropdown

By Boole's inequality (union bound), for any set of events $A_1, \ldots, A_m$:

$$P\left(\bigcup_{i=1}^m A_i\right) \leq \sum_{i=1}^m P(A_i)$$

Let $A_i$ = "test $i$ falsely rejects". If test $i$ uses threshold $\alpha/m$, then $P(A_i) = \alpha/m$ for each true null. Even if all $m$ nulls are true:

$$\text{FWER} = P\left(\bigcup_{i=1}^m A_i\right) \leq \sum_{i=1}^m \frac{\alpha}{m} = \alpha$$

This holds **without any assumption about the correlation structure** between tests.
:::

### Holm's Step-Down Method

Holm's method is uniformly more powerful than Bonferroni (i.e., it rejects at least as many hypotheses) while still controlling FWER.

**Algorithm:**
1. Sort p-values: $p_{(1)} \leq p_{(2)} \leq \ldots \leq p_{(m)}$
2. For the $k$-th smallest p-value, compare against $\alpha / (m - k + 1)$
3. Starting from the smallest: reject as long as $p_{(k)} < \alpha / (m - k + 1)$
4. Stop at the first non-rejection -- all remaining hypotheses are also not rejected.

**Intuition**: The first test uses the Bonferroni threshold $\alpha/m$. But once we reject it, there are only $m-1$ remaining tests to worry about, so the next threshold is $\alpha/(m-1)$, which is less strict. We "step down" through increasingly lenient thresholds.

:::{admonition} Full derivation (click to expand)
:class: dropdown

**Why Holm controls FWER:**

Let $m_0$ be the (unknown) number of true null hypotheses. Consider the true null with the smallest p-value, say $p_{(j)}$.

For a false positive to occur, $p_{(j)}$ must be rejected. But $p_{(j)}$ is compared against $\alpha/(m - j + 1) \leq \alpha/m_0$.

Under the null, $p_{(j)}$ is (stochastically) at most the minimum of $m_0$ uniform$[0,1]$ random variables. The probability that this minimum is below $\alpha/m_0$ is at most $\alpha$ (by the union bound over the $m_0$ true nulls).

**Why Holm dominates Bonferroni:**

Bonferroni uses threshold $\alpha/m$ for all tests. Holm uses $\alpha/m$ for the first (smallest p-value), but $\alpha/(m-1), \alpha/(m-2), \ldots$ for subsequent tests. Since $\alpha/(m-k+1) \geq \alpha/m$ for $k \geq 1$, Holm's thresholds are always at least as lenient as Bonferroni for the second-smallest onward. Any hypothesis rejected by Bonferroni is also rejected by Holm.
:::

### Benjamini-Hochberg (FDR Control)

Instead of controlling the probability of **any** false positive, FDR controls the **expected proportion** of false positives among discoveries:

$$\text{FDR} = E\left[\frac{\text{\# false positives}}{\text{\# rejections}}\right]$$

This is less strict than FWER -- it accepts that some false positives will occur, but keeps them as a controlled fraction of all discoveries.

**Algorithm (Benjamini-Hochberg):**
1. Sort p-values: $p_{(1)} \leq p_{(2)} \leq \ldots \leq p_{(m)}$
2. Find the largest $k$ such that $p_{(k)} \leq \frac{k}{m} \cdot \alpha$
3. Reject all hypotheses $H_{(1)}, \ldots, H_{(k)}$

**Intuition**: If we reject the top $k$ p-values, we expect about $k \cdot (\text{proportion of true nulls})$ of them to be false positives. The BH procedure calibrates $k$ so that this proportion stays below $\alpha$.

**When to use FDR**: Exploratory analyses where you're screening many metrics to find which ones merit further investigation. At QuickCart, we use FDR for secondary metrics (engagement, satisfaction scores) where a few false leads are acceptable.

:::{admonition} Full derivation (click to expand)
:class: dropdown

**Why BH controls FDR at level $\alpha$:**

Let $m_0$ be the number of true nulls. Under independence of the test statistics:

$$\text{FDR} = \frac{m_0}{m} \cdot \alpha \leq \alpha$$

The key insight: p-values for true nulls are uniform$[0,1]$, so they "fill up" the rejection region at a predictable rate. The BH threshold $k\alpha/m$ creates a line in the sorted-p-value plot. True nulls cross this line at rate $\alpha/m$ per hypothesis, giving an expected $m_0 \cdot \alpha/m$ false positives among $k$ rejections.

The formal proof (Benjamini & Hochberg, 1995) uses a clever martingale argument. The result also holds under certain forms of positive dependence (PRDS condition).
:::

In [ ]:
def bonferroni_correction(p_values, alpha=0.05):
    """Apply Bonferroni correction. Returns array of booleans (reject or not)."""
    m = len(p_values)
    adjusted_alpha = alpha / m
    return np.array(p_values) < adjusted_alpha


def holm_correction(p_values, alpha=0.05):
    """Apply Holm's step-down correction. Returns array of booleans (reject or not)."""
    m = len(p_values)
    p_values = np.array(p_values)
    sorted_indices = np.argsort(p_values)
    sorted_p = p_values[sorted_indices]
    
    rejections = np.zeros(m, dtype=bool)
    
    for k in range(m):
        threshold = alpha / (m - k)
        if sorted_p[k] < threshold:
            rejections[sorted_indices[k]] = True
        else:
            # Stop: don't reject this or any remaining
            break
    
    return rejections


def benjamini_hochberg(p_values, alpha=0.05):
    """Apply Benjamini-Hochberg FDR correction. Returns array of booleans (reject or not)."""
    m = len(p_values)
    p_values = np.array(p_values)
    sorted_indices = np.argsort(p_values)
    sorted_p = p_values[sorted_indices]
    
    # Find largest k such that p_(k) <= k/m * alpha
    max_k = -1
    for k in range(m):
        threshold = (k + 1) / m * alpha  # k+1 because 1-indexed in formula
        if sorted_p[k] <= threshold:
            max_k = k
    
    rejections = np.zeros(m, dtype=bool)
    if max_k >= 0:
        # Reject all hypotheses up to and including max_k
        rejections[sorted_indices[:max_k + 1]] = True
    
    return rejections


# Quick demo
demo_pvalues = [0.001, 0.008, 0.03, 0.04, 0.06, 0.12, 0.45, 0.92]
m = len(demo_pvalues)

print(f"P-values: {demo_pvalues}")
print(f"Number of tests: {m}")
print(f"Bonferroni threshold: {0.05/m:.4f}")
print()
print(f"Bonferroni rejections:  {bonferroni_correction(demo_pvalues)}")
print(f"Holm rejections:        {holm_correction(demo_pvalues)}")
print(f"BH (FDR) rejections:    {benjamini_hochberg(demo_pvalues)}")

## Comparison via Simulation

Let's run a realistic simulation: 20 metrics, where 5 have a true effect (alternative is true) and 15 are null (no effect). We'll compare how each correction method trades off between:

- **False Positive Rate**: proportion of null metrics incorrectly declared significant
- **Power (True Positive Rate)**: proportion of real effects successfully detected

In [ ]:
# Simulation: compare correction methods
n_simulations = 5000
n_metrics_total = 20
n_true_effects = 5
n_null = n_metrics_total - n_true_effects
sample_size = 500
effect_size = 0.2  # Cohen's d for true effects
alpha = 0.05

results = {
    'No correction': {'fp': [], 'tp': []},
    'Bonferroni': {'fp': [], 'tp': []},
    'Holm': {'fp': [], 'tp': []},
    'Benjamini-Hochberg': {'fp': [], 'tp': []},
}

for sim in range(n_simulations):
    p_values = []
    is_null = []  # True if the metric has no real effect
    
    # Generate null metrics (no effect)
    for _ in range(n_null):
        control = np.random.normal(0, 1, sample_size)
        treatment = np.random.normal(0, 1, sample_size)  # Same distribution
        _, p = stats.ttest_ind(control, treatment)
        p_values.append(p)
        is_null.append(True)
    
    # Generate metrics with true effects
    for _ in range(n_true_effects):
        control = np.random.normal(0, 1, sample_size)
        treatment = np.random.normal(effect_size, 1, sample_size)  # Shifted
        _, p = stats.ttest_ind(control, treatment)
        p_values.append(p)
        is_null.append(False)
    
    p_values = np.array(p_values)
    is_null = np.array(is_null)
    
    # Apply each correction
    corrections = {
        'No correction': p_values < alpha,
        'Bonferroni': bonferroni_correction(p_values, alpha),
        'Holm': holm_correction(p_values, alpha),
        'Benjamini-Hochberg': benjamini_hochberg(p_values, alpha),
    }
    
    for method, rejections in corrections.items():
        # False positives: rejected among null metrics
        fp_count = np.sum(rejections & is_null)
        # True positives: rejected among real effects
        tp_count = np.sum(rejections & ~is_null)
        results[method]['fp'].append(fp_count)
        results[method]['tp'].append(tp_count)

# Summarize
print(f"Simulation: {n_metrics_total} metrics ({n_true_effects} real effects, {n_null} null)")
print(f"Effect size: Cohen's d = {effect_size}, Sample size: {sample_size} per group")
print(f"{'Method':<22} {'FPR (of nulls)':<18} {'FWER':<12} {'Power':<12} {'Avg FP':<10}")
print("-" * 74)

for method in results:
    fp_arr = np.array(results[method]['fp'])
    tp_arr = np.array(results[method]['tp'])
    fpr = np.mean(fp_arr) / n_null  # Average FP rate per null metric
    fwer = np.mean(fp_arr > 0)  # P(at least one FP)
    power = np.mean(tp_arr) / n_true_effects  # Average power per real effect
    avg_fp = np.mean(fp_arr)
    print(f"{method:<22} {fpr:<18.4f} {fwer:<12.4f} {power:<12.4f} {avg_fp:<10.2f}")

In [ ]:
# Visualize the trade-off
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

methods = list(results.keys())
fwer_values = [np.mean(np.array(results[m]['fp']) > 0) for m in methods]
power_values = [np.mean(results[m]['tp']) / n_true_effects for m in methods]

colors = ['#d62728', '#2ca02c', '#1f77b4', '#ff7f0e']

# FWER comparison
axes[0].bar(methods, fwer_values, color=colors, alpha=0.8)
axes[0].axhline(y=0.05, color='black', linestyle='--', label='Target alpha=0.05')
axes[0].set_ylabel('FWER')
axes[0].set_title('Family-Wise Error Rate')
axes[0].legend()
axes[0].set_ylim(0, 0.7)
axes[0].tick_params(axis='x', rotation=15)

# Power comparison
axes[1].bar(methods, power_values, color=colors, alpha=0.8)
axes[1].set_ylabel('Power (True Positive Rate)')
axes[1].set_title('Statistical Power')
axes[1].set_ylim(0, 1.0)
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.suptitle('Multiple Testing Corrections: Error Rate vs Power Trade-off', 
             y=1.02, fontsize=13, fontweight='bold')
plt.show()

## When to Use Which Method

| Scenario | Method | Why |
|----------|--------|-----|
| Guardrail metrics (revenue, crash rate) | **Bonferroni** or **Holm** | A single false positive could block a good launch or greenlight a bad one. FWER control is essential. |
| Few critical metrics (2-5) | **Holm** | More powerful than Bonferroni, still strict FWER control. Holm dominates Bonferroni. |
| Exploratory metrics (20+ secondary KPIs) | **Benjamini-Hochberg** | You want to discover real signals without being paralyzed by conservatism. A few false leads are acceptable. |
| Single primary metric | **No correction needed** | If you pre-registered one primary metric, test it at $\alpha$. |

At QuickCart, we use a **tiered approach**:
- **Tier 1** (ship/no-ship decision): 2-3 primary metrics with Holm correction
- **Tier 2** (guardrails): 5-8 safety metrics with Bonferroni (conservative is fine here)
- **Tier 3** (learning): 10-20 exploratory metrics with Benjamini-Hochberg

---

# Part 2: Experiment Infrastructure -- User Splitting

## The Challenge

QuickCart runs 50+ experiments concurrently. We need a system that:
1. **Deterministically** assigns users to groups (same user always gets same experience)
2. **Doesn't require storing assignments** (we have 10M+ users)
3. **Handles conflicts** (experiments that can't overlap, e.g., two different checkout flows)
4. **Distributes evenly** across control/treatment groups

## Hash-Based Assignment

The core idea: use a hash function to convert `user_id` into a number, then use that number to determine group assignment. Because hash functions are deterministic, the same user always gets the same result.

```python
slot = hash(user_id) % num_slots
group = hash(user_id + experiment_id) % 2  # 0=control, 1=treatment
```

## Double Hashing for Slot Allocation

We use a **two-level hashing** scheme:

1. **First hash** (`salt_one + user_id`): Maps each user to a **slot** (0 to `count_slots-1`). This determines which experiments the user participates in.

2. **Second hash** (`salt_two + user_id + experiment_id`): Maps each user to **control or treatment** within a specific experiment.

Why two hashes? If we used the same hash for both decisions, the slot assignment and group assignment would be correlated, creating bias.

## Conflict-Aware Slot Allocation

Some experiments **cannot overlap**. For example:
- Experiment A: Blue "Buy Now" button
- Experiment B: Green "Buy Now" button

A user cannot be in both. We handle this by assigning conflicting experiments to **different slots**, so no user sees both.

**Algorithm:**
1. Sort experiments by number of conflicts (most constrained first)
2. For each experiment, find slots that don't conflict with already-assigned experiments
3. Among available slots, choose the **least loaded** one (fewest experiments assigned)

In [ ]:
class ABSplitter:
    """
    Hash-based experiment assignment system with conflict-aware slot allocation.
    
    Uses double hashing:
    - First hash (salt_one + user_id): assigns user to a slot
    - Second hash (salt_two + user_id + experiment_id): assigns user to control/treatment
    """
    
    def __init__(self, count_slots, salt_one, salt_two):
        """
        Parameters
        ----------
        count_slots : int
            Number of traffic slots (e.g., 100 for 1% granularity)
        salt_one : str
            Salt for slot assignment hash
        salt_two : str
            Salt for group assignment hash
        """
        self.count_slots = count_slots
        self.salt_one = salt_one
        self.salt_two = salt_two
        # slot_id -> list of experiment names assigned to that slot
        self.slot_assignments = {i: [] for i in range(count_slots)}
        # experiment_name -> list of slot_ids
        self.experiment_slots = {}
    
    def _hash(self, value, modulo):
        """Deterministic hash using MD5."""
        return int(hashlib.md5(str.encode(value)).hexdigest(), 16) % modulo
    
    def split_experiments(self, experiments):
        """
        Allocate experiments to slots, respecting conflicts.
        
        Parameters
        ----------
        experiments : list of dict
            Each dict has:
            - 'name': str, experiment identifier
            - 'num_slots': int, how many slots this experiment needs
            - 'conflicts': list of str, names of experiments that cannot overlap
        """
        # Sort by number of conflicts (most constrained first)
        sorted_experiments = sorted(experiments, key=lambda x: len(x['conflicts']), reverse=True)
        
        for exp in sorted_experiments:
            name = exp['name']
            num_slots_needed = exp['num_slots']
            conflicts = set(exp['conflicts'])
            
            # Find slots that are NOT used by any conflicting experiment
            blocked_slots = set()
            for conflict_name in conflicts:
                if conflict_name in self.experiment_slots:
                    blocked_slots.update(self.experiment_slots[conflict_name])
            
            available_slots = [s for s in range(self.count_slots) if s not in blocked_slots]
            
            if len(available_slots) < num_slots_needed:
                raise ValueError(
                    f"Experiment '{name}' needs {num_slots_needed} slots but only "
                    f"{len(available_slots)} are available (conflicts block {len(blocked_slots)} slots)"
                )
            
            # Among available slots, pick the least loaded ones
            available_slots.sort(key=lambda s: len(self.slot_assignments[s]))
            assigned_slots = available_slots[:num_slots_needed]
            
            # Record assignments
            self.experiment_slots[name] = assigned_slots
            for slot in assigned_slots:
                self.slot_assignments[slot].append(name)
        
        return self.experiment_slots
    
    def process_user(self, user_id):
        """
        Determine which experiments a user participates in and their group.
        
        Parameters
        ----------
        user_id : str
            Unique user identifier
            
        Returns
        -------
        dict
            {experiment_name: 'control' or 'treatment'} for each experiment
            the user is assigned to
        """
        # First hash: which slot is this user in?
        user_slot = self._hash(self.salt_one + user_id, self.count_slots)
        
        # Get experiments running in this user's slot
        experiments_for_user = self.slot_assignments[user_slot]
        
        # Second hash: for each experiment, determine control vs treatment
        assignments = {}
        for exp_name in experiments_for_user:
            group_hash = self._hash(self.salt_two + user_id + exp_name, 2)
            assignments[exp_name] = 'treatment' if group_hash == 1 else 'control'
        
        return assignments

In [ ]:
# Demo: QuickCart experiment setup
splitter = ABSplitter(count_slots=20, salt_one='quickcart_2024_slot_', salt_two='quickcart_2024_group_')

experiments = [
    {'name': 'blue_checkout_button', 'num_slots': 5, 'conflicts': ['green_checkout_button']},
    {'name': 'green_checkout_button', 'num_slots': 5, 'conflicts': ['blue_checkout_button']},
    {'name': 'new_search_ranking', 'num_slots': 10, 'conflicts': []},
    {'name': 'free_shipping_banner', 'num_slots': 8, 'conflicts': []},
    {'name': 'one_click_reorder', 'num_slots': 4, 'conflicts': []},
]

allocations = splitter.split_experiments(experiments)

print("Experiment Slot Allocations:")
print("=" * 50)
for exp_name, slots in allocations.items():
    print(f"  {exp_name:<25} -> slots {sorted(slots)}")

print("\nSlot Utilization:")
print("-" * 50)
for slot_id in range(splitter.count_slots):
    exps = splitter.slot_assignments[slot_id]
    print(f"  Slot {slot_id:2d}: {exps if exps else '(empty)'}")

In [ ]:
# Process some users to see their assignments
print("User Assignments:")
print("=" * 60)

sample_users = [f'user_{i}' for i in range(10)]

for user_id in sample_users:
    assignments = splitter.process_user(user_id)
    slot = splitter._hash(splitter.salt_one + user_id, splitter.count_slots)
    if assignments:
        print(f"  {user_id} (slot {slot:2d}): {assignments}")
    else:
        print(f"  {user_id} (slot {slot:2d}): (no experiments in this slot)")

In [ ]:
# Verify even distribution across groups
n_users = 100000
group_counts = {exp['name']: {'control': 0, 'treatment': 0} for exp in experiments}
participation_counts = {exp['name']: 0 for exp in experiments}

for i in range(n_users):
    user_id = f'user_{i}'
    assignments = splitter.process_user(user_id)
    for exp_name, group in assignments.items():
        group_counts[exp_name][group] += 1
        participation_counts[exp_name] += 1

print(f"Distribution check across {n_users:,} users:")
print(f"{'Experiment':<25} {'Participants':<14} {'Control':<10} {'Treatment':<10} {'Ratio':<8}")
print("-" * 67)
for exp_name in group_counts:
    c = group_counts[exp_name]['control']
    t = group_counts[exp_name]['treatment']
    total = c + t
    expected_pct = len(splitter.experiment_slots[exp_name]) / splitter.count_slots * 100
    ratio = t / c if c > 0 else float('inf')
    print(f"  {exp_name:<25} {total:<14,} {c:<10,} {t:<10,} {ratio:<8.3f}")

print("\n(Ratio close to 1.0 indicates balanced control/treatment split)")

In [ ]:
# Verify that conflicting experiments never overlap for any user
n_users_check = 50000
conflict_violations = 0

for i in range(n_users_check):
    user_id = f'user_{i}'
    assignments = splitter.process_user(user_id)
    assigned_experiments = set(assignments.keys())
    
    # Check: user should never be in both blue_checkout_button and green_checkout_button
    if 'blue_checkout_button' in assigned_experiments and 'green_checkout_button' in assigned_experiments:
        conflict_violations += 1

print(f"Conflict check across {n_users_check:,} users:")
print(f"  Users in both conflicting checkout experiments: {conflict_violations}")
print(f"  Conflict isolation: {'PASSED' if conflict_violations == 0 else 'FAILED'}")

---

## Summary

| Topic | Key Takeaway |
|-------|-------------|
| **Multiple Testing Problem** | Testing $m$ metrics at $\alpha$ gives FWER $\approx 1-(1-\alpha)^m$. With 20 metrics, 64% chance of a false positive. |
| **Bonferroni** | Divide $\alpha$ by $m$. Simple, valid always, but conservative. Use for guardrails. |
| **Holm Step-Down** | Uniformly more powerful than Bonferroni. Controls FWER. Use as default for critical metrics. |
| **Benjamini-Hochberg** | Controls FDR (proportion of false discoveries). More discoveries at cost of some false positives. Use for exploratory metrics. |
| **Hash-Based Splitting** | MD5 hash of `salt + user_id` gives deterministic, storage-free assignment. |
| **Double Hashing** | Separate hashes for slot (which experiments) and group (control/treatment) prevent correlation. |
| **Conflict-Aware Allocation** | Sort by conflicts (most constrained first), assign to least-loaded available slots. |